<a href="https://www.kaggle.com/code/byydbck/llmresearch1?scriptVersionId=364050386" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
!pip install unsloth #optimizes the LLM training execution. this project is so much faster with unsloth.
!pip install --no-deps --upgrade "torchao>=0.16.0" #optimizes tensor operations,reduce memory footprint
!pip install transformers==4.57.6 #for tokenization
!pip install --no-deps trl==0.22.2 #transformer reinforcement learning 
#used --no-deps for trl and torchao because we didn't want them to automatically update the dependencies for version errors.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.2/30.2 MB 50.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 31.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 106.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 66.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 88.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 68.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 64.7 MB/s  0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11/11 [unsloth]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 28.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 13.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 53.5 MB/s  0:00:00
  Attempting uninstall: hu

In [2]:
from unsloth import FastLanguageModel
import torch

"""loaded the qwen2.5-7B model, assigned 2048 as the maximum token context length for one sequence
enabled 4-bit quantization and automatically detected optimal data type
"""
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-7B",
    max_seq_length=2048, 
    dtype=None,
    load_in_4bit=True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.10.11: Fast Qwen2 patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Unsloth: Not splitting across GPUs; the model fits on cuda:0 with room to train. Pass device_map = "unsloth_balanced" to split it anyway.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [3]:
FastLanguageModel.for_inference(model) #switched the model to inference mode for faster generation


#defined the standart alpaca prompt format
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""



#tokenized the prompt and moved tensors to the CUDA GPU
FastLanguageModel.for_inference(model) 
inputs = tokenizer(
[
    alpaca_prompt.format(
        "Continue the fibonacci sequence.", #instruction
        "1, 1, 2, 3, 5, 8", #input
        "", #output is empty for the model to fill
    )
], return_tensors = "pt").to("cuda") #return as python tensors


#decode back to token ids -> strings
from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
outputs = model.generate(
    **inputs,         
    max_new_tokens=64, 
    use_cache=True,     
    streamer=text_streamer #streams the responses in real-time instead of making you wait without outcome
)
tokenizer.batch_decode(outputs)
#these codes were taken from the Unsloth Qwen2.5 7B Google Colab documentation. 
#https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen2_(7B)-Alpaca.ipynb#scrollTo=_g5uNxjbNx_2


Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Continue the fibonacci sequence.

### Input:
1, 1, 2, 3, 5, 8

### Response:
13, 21, 34, 55, 89, 144, 233, 377, 610, 987, 1597, 2584, 4181, 6


['Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\nContinue the fibonacci sequence.\n\n### Input:\n1, 1, 2, 3, 5, 8\n\n### Response:\n13, 21, 34, 55, 89, 144, 233, 377, 610, 987, 1597, 2584, 4181, 6']

In [4]:
from datasets import load_dataset
dataset = load_dataset("yahma/alpaca-cleaned", split="train") #imported the dataset from hugging face using datasets lib

README.md: 0.00B [00:00, ?B/s]

alpaca_data_cleaned.json:   0%|          | 0.00/44.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/51760 [00:00<?, ? examples/s]

In [5]:
print(dataset[0])
#as we can see there are outputs,inputs and instructions. not all data need inputs, so for it to be blank is considered normal.

{'output': '1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.\n\n2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.\n\n3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.', 'input': '', 'instruction': 'Give three tips for staying healthy.'}


In [6]:
test_dataset = dataset.shuffle(seed=42).select(range(200))
print(f"test set examples: {len(test_dataset)}")
print({test_dataset[0]['instruction']})
#"all assignment options require at least 200 held-out test examples." we will be using the same test set for each evaluation (for B1,B2 and B3)

test set examples: 200
{'Rearrange the following sentence to make the sentence more interesting.'}


In [7]:
import pandas as pd

# for b1: zero-shot 
samples = 200
b1_results = []

for i in range(samples):
    sample = test_dataset[i]
    instruction = sample["instruction"]
    input_text = sample["input"]
    output_text = sample["output"]
    
    prompt_b1 = alpaca_prompt.format(instruction, input_text, "")
    inputs_b1 = tokenizer([prompt_b1], return_tensors="pt").to("cuda")
    
    outputs_b1 = model.generate(
        **inputs_b1, 
        max_new_tokens=256, 
        use_cache=True,
        do_sample=False
    )
    
    # extract newly generated tokens
    input_len = inputs_b1["input_ids"].shape[1]
    pred_b1 = tokenizer.decode(outputs_b1[0][input_len:], skip_special_tokens=True).strip()
    
    # stop the model from generating new instructions
    pred_b1 = pred_b1.split("### Instruction:")[0].strip()
    
    # required format: sample id, condition, prediction, reference
    b1_results.append({
        "sample_id": i,
        "condition": "b1_zero_shot",
        "prediction": pred_b1,
        "reference": output_text
    })

In [8]:
df_b1 = pd.DataFrame(b1_results)
df_b1.to_csv("b1_predictions.csv", index=False)

pd.set_option('display.max_colwidth', None)
df_b1.head(5)

,sample_id,condition,prediction,reference
0,0,b1_zero_shot,"She left the party early, but not before she had a great time dancing and chatting with her friends.","Early, she left the party."
1,1,b1_zero_shot,"To find the sum of all values of x such that f(x) = 0, we need to solve the equation f(x) = 0 for both cases of the piecewise function.\n\nCase 1: x ≤ 1\nf(x) = -x - 3\nSetting f(x) = 0, we get:\n-x - 3 = 0\nSolving for x, we get:\nx = -3\n\nCase 2: x > 1\nf(x) = x/2 + 1\nSetting f(x) = 0, we get:\nx/2 + 1 = 0\nSolving for x, we get:\nx = -2\n\nTherefore, the sum of all values of x such that f(x) = 0 is -3 + (-2) = -5.","We solve the equation f(x) = 0 on the domains x ≤ 1 and x > 1.\n\nIf x ≤ 1, then f(x) = -x - 3, so we want to solve -x - 3 = 0. The solution is x = -3, which satisfies x ≤ 1.\n\nIf x > 1, then f(x) = x/2 + 1, so we want to solve x/2 + 1 = 0. The solution is x = -2, but this value does not satisfy x > 1.\n\nTherefore, the only solution is x = -3."
2,2,b1_zero_shot,"A warm summer day,\nGolden sun shining bright,\nNature's beauty awakes.","Warm breeze on my face\nEndless sun brings joy and peace\nSummer, please don't fade"
3,3,b1_zero_shot,"There are several methods that can be used to improve the accuracy of machine learning models, including:\n\n1. Feature engineering: This involves selecting and transforming the input features to make them more relevant and informative for the model. This can be done by removing irrelevant features, scaling or normalizing the features, or creating new features that capture important patterns in the data.\n\n2. Hyperparameter tuning: This involves adjusting the parameters of the model to optimize its performance. This can be done using techniques such as grid search, random search, or Bayesian optimization.\n\n3. Ensemble methods: This involves combining multiple models to improve their accuracy. This can be done using techniques such as bagging, boosting, or stacking.\n\n4. Regularization: This involves adding a penalty term to the loss function to prevent overfitting. This can be done using techniques such as L1 or L2 regularization, dropout, or early stopping.\n\n5. Data augmentation: This involves generating new training data by applying transformations to the existing data. This can be done using techniques such as rotation, scaling, or flipping.\n\n6. Transfer learning: This involves using a pre-trained model as a starting point for a new task. This can be done using techniques such as fine-tuning or feature extraction.\n\n7. Active learning:","Here are several methods to improve the accuracy of machine learning models:\n\n1. Gathering and cleaning more data: The more data the model is trained on, the more patterns it can recognize, and the better it can generalize to new data. It’s also important to clean the data to handle missing, inconsistent or duplicate information.\n\n2. Data preprocessing: Several preprocessing techniques like normalization, standardization, encoding, and feature selection can improve model accuracy by making the data more suitable for the learning algorithm.\n\n3. Selecting the right algorithm: Different algorithms are suited to different types of data and problems. Experimenting with various algorithms, including ensemble methods that combine several algorithms, can boost accuracy.\n\n4. Hyperparameter tuning: Hyperparameters are the fixed parameters that control the behavior of the learning algorithm. Tuning them can sometimes result in large improvements in accuracy.\n\n5. Regularization: Regularization is a technique to prevent overfitting by adding a penalty term to the loss function. It encourages the model to use all the input features in a more balanced way, thus resulting in better generalization on new data.\n\n6. Cross-validation: Cross-validation is a technique to evaluate how well the model generalizes on new, unseen data. By using cross-validation to help choose the best combination of preprocessing techniques, algorithms, and 

In [9]:
import pandas as pd

# for b2: chain-of-thought
samples = 200
b2_results = []

for i in range(samples):
    sample = test_dataset[i]
    instruction = sample["instruction"]
    input_text = sample["input"]
    output_text = sample["output"]
    
    # cot prompting
    cot_instruction = instruction + " Let's think step by step to arrive at the correct answer."
    
    prompt_b2 = alpaca_prompt.format(cot_instruction, input_text, "")
    inputs_b2 = tokenizer([prompt_b2], return_tensors="pt").to("cuda")
    
    outputs_b2 = model.generate(
        **inputs_b2, 
        max_new_tokens=256,  
        use_cache=True,
        do_sample=False
    )
    
    # extract newly generated tokens
    input_len = inputs_b2["input_ids"].shape[1]
    pred_b2 = tokenizer.decode(outputs_b2[0][input_len:], skip_special_tokens=True).strip()
    
    # stop the model from generating new instructions
    pred_b2 = pred_b2.split("### Instruction:")[0].strip()
    
    b2_results.append({
        "sample_id": i,
        "condition": "b2_cot",
        "prediction": pred_b2,
        "reference": output_text
    })

In [10]:
df_b2 = pd.DataFrame(b2_results)
df_b2.to_csv("b2_predictions.csv", index=False)

pd.set_option('display.max_colwidth', None)
df_b2.head(5)

,sample_id,condition,prediction,reference
0,0,b2_cot,"She left the party early, but not before she had a great time dancing and chatting with her friends.","Early, she left the party."
1,1,b2_cot,"To find the sum of all values of x such that f(x) = 0, we need to consider both cases of the piecewise function:\n\nCase 1: x ≤ 1\nIn this case, f(x) = -x - 3. Setting this equal to 0, we get:\n-x - 3 = 0\nx = -3\n\nCase 2: x > 1\nIn this case, f(x) = x/2 + 1. Setting this equal to 0, we get:\nx/2 + 1 = 0\nx = -2\n\nHowever, since x > 1 in this case, x = -2 is not a valid solution.\n\nTherefore, the only valid solution is x = -3.\n\nThe sum of all values of x such that f(x) = 0 is -3.","We solve the equation f(x) = 0 on the domains x ≤ 1 and x > 1.\n\nIf x ≤ 1, then f(x) = -x - 3, so we want to solve -x - 3 = 0. The solution is x = -3, which satisfies x ≤ 1.\n\nIf x > 1, then f(x) = x/2 + 1, so we want to solve x/2 + 1 = 0. The solution is x = -2, but this value does not satisfy x > 1.\n\nTherefore, the only solution is x = -3."
2,2,b2_cot,"A warm summer day,\nGolden sun shining bright,\nNature's symphony.","Warm breeze on my face\nEndless sun brings joy and peace\nSummer, please don't fade"
3,3,b2_cot,"To improve the accuracy of machine learning models, we can use the following methods:\n\n1. Data preprocessing: This involves cleaning and transforming the data to make it more suitable for the model. This can include removing outliers, handling missing values, and scaling the data.\n\n2. Feature selection: This involves selecting the most relevant features for the model to improve its accuracy. This can be done using techniques such as correlation analysis, feature importance ranking, and feature selection algorithms.\n\n3. Model selection: This involves choosing the most appropriate model for the task at hand. This can be done using techniques such as cross-validation, grid search, and ensemble methods.\n\n4. Hyperparameter tuning: This involves adjusting the parameters of the model to improve its accuracy. This can be done using techniques such as grid search, random search, and Bayesian optimization.\n\n5. Regularization: This involves adding a penalty term to the model to prevent overfitting. This can be done using techniques such as L1 and L2 regularization.\n\n6. Ensemble methods: This involves combining multiple models to improve their accuracy. This can be done using techniques such as bagging, boosting, and stacking.\n\n7. Transfer learning: This involves using pre-trained models to improve the accuracy of the model. This can be done using techniques","Here are several methods to improve the accuracy of machine learning models:\n\n1. Gathering and cleaning more data: The more data the model is trained on, the more patterns it can recognize, and the better it can generalize to new data. It’s also important to clean the data to handle missing, inconsistent or duplicate information.\n\n2. Data preprocessing: Several preprocessing techniques like normalization, standardization, encoding, and feature selection can improve model accuracy by making the data more suitable for the learning algorithm.\n\n3. Selecting the right algorithm: Different algorithms are suited to different types of data and problems. Experimenting with various algorithms, including ensemble methods that combine several algorithms, can boost accuracy.\n\n4. Hyperparameter tuning: Hyperparameters are the fixed parameters that control the behavior of the learning algorithm. Tuning them can sometimes result in large improvements in accuracy.\n\n5. Regularization: Regularization is a technique to prevent overfitting by adding a penalty term to the loss function. It encourages the model to use all the input features in a more balanced way, thus resulting in better generalization on new data.\n\n6. Cross-validation: Cross-validation is a technique to evaluate how well the model generalizes on new, unseen data. By using cross-validation to help choose the best combin

In [11]:
import pandas as pd

# for b3: role/persona (instead of pal)
samples = 200
b3_results = []

for i in range(samples):
    sample = test_dataset[i]
    instruction = sample["instruction"]
    input_text = sample["input"]
    output_text = sample["output"]
    
    # persona prompting
    persona_instruction = "You are a helpful and precise expert. " + instruction
    
    prompt_b3 = alpaca_prompt.format(persona_instruction, input_text, "")
    inputs_b3 = tokenizer([prompt_b3], return_tensors="pt").to("cuda")
    
    outputs_b3 = model.generate(
        **inputs_b3, 
        max_new_tokens=256, 
        use_cache=True,
        do_sample=False
    )
    
    # extract newly generated tokens
    input_len = inputs_b3["input_ids"].shape[1]
    pred_b3 = tokenizer.decode(outputs_b3[0][input_len:], skip_special_tokens=True).strip()
    
    # stop the model from generating new instructions
    pred_b3 = pred_b3.split("### Instruction:")[0].strip()
    
    b3_results.append({
        "sample_id": i,
        "condition": "b3_persona",
        "prediction": pred_b3,
        "reference": output_text
    })

In [12]:
df_b3 = pd.DataFrame(b3_results)
df_b3.to_csv("b3_predictions.csv", index=False)

pd.set_option('display.max_colwidth', None)
df_b3.head(5)

,sample_id,condition,prediction,reference
0,0,b3_persona,"She left the party early, but not before she had a chance to say goodbye to everyone.","Early, she left the party."
1,1,b3_persona,"To find the sum of all values of x such that f(x) = 0, we need to solve for x in each of the two cases given by the piecewise function:\n\nCase 1: x ≤ 1\nf(x) = -x - 3\nSetting f(x) = 0, we get:\n-x - 3 = 0\nSolving for x, we get:\nx = -3\n\nCase 2: x > 1\nf(x) = x/2 + 1\nSetting f(x) = 0, we get:\nx/2 + 1 = 0\nSolving for x, we get:\nx = -2\n\nTherefore, the sum of all values of x such that f(x) = 0 is -3 + (-2) = -5.","We solve the equation f(x) = 0 on the domains x ≤ 1 and x > 1.\n\nIf x ≤ 1, then f(x) = -x - 3, so we want to solve -x - 3 = 0. The solution is x = -3, which satisfies x ≤ 1.\n\nIf x > 1, then f(x) = x/2 + 1, so we want to solve x/2 + 1 = 0. The solution is x = -2, but this value does not satisfy x > 1.\n\nTherefore, the only solution is x = -3."
2,2,b3_persona,"A warm breeze blows,\nGolden sun shines bright,\nSummer day awakes.","Warm breeze on my face\nEndless sun brings joy and peace\nSummer, please don't fade"
3,3,b3_persona,"There are several methods that can be used to improve the accuracy of machine learning models, including:\n\n1. Feature engineering: This involves selecting and transforming the input features to make them more relevant and informative for the model. This can be done by removing irrelevant features, scaling or normalizing the features, or creating new features that capture important patterns in the data.\n\n2. Hyperparameter tuning: This involves adjusting the parameters of the model to optimize its performance. This can be done using techniques such as grid search, random search, or Bayesian optimization.\n\n3. Ensemble methods: This involves combining multiple models to improve their accuracy. This can be done using techniques such as bagging, boosting, or stacking.\n\n4. Regularization: This involves adding a penalty term to the loss function to prevent overfitting. This can be done using techniques such as L1 or L2 regularization, dropout, or early stopping.\n\n5. Data augmentation: This involves generating new training data by applying transformations to the existing data. This can be done using techniques such as rotation, scaling, or flipping.\n\n6. Transfer learning: This involves using a pre-trained model as a starting point for a new task. This can be done using techniques such as fine-tuning or feature extraction.\n\n7. Active learning:","Here are several methods to improve the accuracy of machine learning models:\n\n1. Gathering and cleaning more data: The more data the model is trained on, the more patterns it can recognize, and the better it can generalize to new data. It’s also important to clean the data to handle missing, inconsistent or duplicate information.\n\n2. Data preprocessing: Several preprocessing techniques like normalization, standardization, encoding, and feature selection can improve model accuracy by making the data more suitable for the learning algorithm.\n\n3. Selecting the right algorithm: Different algorithms are suited to different types of data and problems. Experimenting with various algorithms, including ensemble methods that combine several algorithms, can boost accuracy.\n\n4. Hyperparameter tuning: Hyperparameters are the fixed parameters that control the behavior of the learning algorithm. Tuning them can sometimes result in large improvements in accuracy.\n\n5. Regularization: Regularization is a technique to prevent overfitting by adding a penalty term to the loss function. It encourages the model to use all the input features in a more balanced way, thus resulting in better generalization on new data.\n\n6. Cross-validation: Cross-validation is a technique to evaluate how well the model generalizes on new, unseen data. By using cross-validation to help choose the best combination of preprocessing techniques, algorithms, and hyperparameters, it’s possi

In [13]:
!pip install -q evaluate rouge_score bert_score

import evaluate

# load metrics
rouge = evaluate.load('rouge')
bertscore = evaluate.load('bertscore')

# calculate b1 metrics
rouge_b1 = rouge.compute(predictions=df_b1["prediction"].tolist(), references=df_b1["reference"].tolist())
bert_b1 = bertscore.compute(predictions=df_b1["prediction"].tolist(), references=df_b1["reference"].tolist(), lang="en")
bert_f1_b1 = sum(bert_b1['f1']) / len(bert_b1['f1'])

# calculate b2 metrics
rouge_b2 = rouge.compute(predictions=df_b2["prediction"].tolist(), references=df_b2["reference"].tolist())
bert_b2 = bertscore.compute(predictions=df_b2["prediction"].tolist(), references=df_b2["reference"].tolist(), lang="en")
bert_f1_b2 = sum(bert_b2['f1']) / len(bert_b2['f1'])

# calculate b3 metrics
rouge_b3 = rouge.compute(predictions=df_b3["prediction"].tolist(), references=df_b3["reference"].tolist())
bert_b3 = bertscore.compute(predictions=df_b3["prediction"].tolist(), references=df_b3["reference"].tolist(), lang="en")
bert_f1_b3 = sum(bert_b3['f1']) / len(bert_b3['f1'])

# print scores
print("--- b1 zero-shot scores ---")
print(f"rouge-l: {rouge_b1['rougeL']:.4f}")
print(f"bertscore f1: {bert_f1_b1:.4f}\n")

print("--- b2 cot scores ---")
print(f"rouge-l: {rouge_b2['rougeL']:.4f}")
print(f"bertscore f1: {bert_f1_b2:.4f}\n")

print("--- b3 persona scores ---")
print(f"rouge-l: {rouge_b3['rougeL']:.4f}")
print(f"bertscore f1: {bert_f1_b3:.4f}")

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


--- b1 zero-shot scores ---
rouge-l: 0.3741
bertscore f1: 0.9015

--- b2 cot scores ---
rouge-l: 0.2671
bertscore f1: 0.8761

--- b3 persona scores ---
rouge-l: 0.3699
bertscore f1: 0.9007


In [14]:
import evaluate

# load bleu metric
bleu = evaluate.load('bleu')

# calculate bleu for all conditions
bleu_b1 = bleu.compute(predictions=df_b1["prediction"].tolist(), references=df_b1["reference"].tolist())
bleu_b2 = bleu.compute(predictions=df_b2["prediction"].tolist(), references=df_b2["reference"].tolist())
bleu_b3 = bleu.compute(predictions=df_b3["prediction"].tolist(), references=df_b3["reference"].tolist())

# print bleu scores
print("--- bleu scores ---")
print(f"b1 zero-shot: {bleu_b1['bleu']:.4f}")
print(f"b2 cot: {bleu_b2['bleu']:.4f}")
print(f"b3 persona: {bleu_b3['bleu']:.4f}")

--- bleu scores ---
b1 zero-shot: 0.1461
b2 cot: 0.1113
b3 persona: 0.1502


In [15]:
""" 
thank you for your time. i also would like to add additional information based on the project:
- i evaluated b1, b2 and b3 (persona) prompting techniques separately instead of adding a loop and doing them together and faster. 
  for my paper and other required documents, i found it to be easier to understand. this may not be the best way to code in real-life problems
  because it is probably slower.
- bert, bleu and rouge have been used for scoring but bleu is not a suitable validation metric for alpaca dataset. i just added to compare different
  metrics.
- i used ai to understand the codes that i retrieved from sources and documentations. i also gave sources to ai and made it adjust the codes 
  for my project.
"""

' \nthank you for your time. i also would like to add additional information based on the project:\n- i evaluated b1, b2 and b3 (persona) prompting techniques separately instead of adding a loop and doing them together and faster. \n  for my paper and other required documents, i found it to be easier to understand. this may not be the best way to code in real-life problems\n  because it is probably slower.\n- bert, bleu and rouge have been used for scoring but bleu is not a suitable validation metric for alpaca dataset. i just added to compare different\n  metrics.\n- i used ai to understand the codes that i retrieved from sources and documentations. i also gave sources to ai and made it adjust the codes \n  for my project.\n'